# Day 1.4 — Tool Calling

A model generates text. It cannot open your files, call an API, or run your Python. What
it *can* do is emit a structured request saying "I would like the calculator, with these
arguments". Your application decides what happens next:

```text
User -> model REQUESTS a tool -> Python validates -> Python executes -> result goes back to the model
```

That arrow in the middle is the whole lesson. Nothing runs until your code runs it.

## Before you begin

### Learning outcomes

- Describe a Python function to a model with a JSON tool schema.
- Read a `tool_calls` request and see that it is only a request.
- Validate the arguments, execute the function, and feed the observation back.

Architecture reference: [D03](../../diagrams/source/day_01.md).

### Expected observation

The model returns a tool name and an arguments string. The calculator runs only after our
own validation step, and an invalid argument leaves the function completely untouched.

In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — Build a calculator that is safe to expose

Never call `eval()` on text a model produced. Below is a **simplified teaching version**:
it parses the expression into a syntax tree and walks it, allowing numbers and four
operators and nothing else. It is short enough to read in one sitting, which is its job.

The version the Day 1 project actually ships is
`src/research_agent/tools.py::calculate` — same idea, more operators, an exponent guard,
and unit tests. We compare the two in Step 2.

In [ ]:
import ast, json, operator

# --- Simplified teaching version -----------------------------------------------
BINARY = {ast.Add: operator.add, ast.Sub: operator.sub,
          ast.Mult: operator.mul, ast.Div: operator.truediv}
UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def evaluate_node(node):
    """Walk one node of the parsed expression. Anything unexpected is refused."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)                       # a plain number
    if isinstance(node, ast.BinOp) and type(node.op) in BINARY:
        return BINARY[type(node.op)](evaluate_node(node.left), evaluate_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY:
        return UNARY[type(node.op)](evaluate_node(node.operand))
    raise ValueError("Only basic arithmetic is allowed")

def teaching_calculator(expression: str) -> str:
    value = evaluate_node(ast.parse(expression, mode="eval").body)
    return str(int(value)) if value.is_integer() else str(value)

print("12 * 7        ->", teaching_calculator("12 * 7"))
print("(3 + 4) / 2   ->", teaching_calculator("(3 + 4) / 2"))

for dangerous in ["__import__('os').getcwd()", "open('secrets.txt').read()"]:
    try:
        teaching_calculator(dangerous)
    except (ValueError, SyntaxError) as error:
        print(f"{dangerous[:26]:28} -> refused: {error}")

### Step 2 — Compare it with the project version

Two differences worth noticing: the shipped version supports more operators, and it caps
the exponent so `2 ** 999999` cannot freeze the kernel. Reading the difference is how you
learn what "hardened" means in practice.

In [ ]:
# PROJECT_ROOT/src is already on sys.path thanks to the course setup cell.
from research_agent.tools import calculate as project_calculator

cases = ["12 * 7", "2 ** 8", "2 ** 999", "__import__('os').getcwd()"]
print(f"{'expression':28} {'teaching version':24} project version")
for expression in cases:
    try:
        teaching = teaching_calculator(expression)
    except Exception as error:
        teaching = f"refused ({type(error).__name__})"
    try:
        project = project_calculator(expression)
    except Exception as error:
        project = f"refused ({type(error).__name__})"
    print(f"{expression:28} {teaching:24} {project}")

print()
print("From here on, use research_agent.tools.calculate. The teaching version stays")
print("in this notebook only so you can read every line of it.")

### Step 3 — Describe the tool to the model

The model never sees your function. It sees this dictionary: a name, a sentence of
description, and a JSON Schema for the arguments. Write the description as if for a
colleague — it is the only thing that tells the model *when* to reach for the tool.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class CalculatorArguments(BaseModel):
    """Our own second line of defence: what we accept, regardless of what was sent."""
    expression: str = Field(min_length=1, max_length=100)

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression instead of calculating mentally.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
            "additionalProperties": False,      # no invented extra arguments
        },
    },
}

print(json.dumps(calculator_tool, indent=2))

### Step 4 — Ask the model, and inspect what comes back

We normalise both routes to plain dictionaries so the rest of the notebook reads the same
whether you are live or mocked.

In [ ]:
MOCK_TOOL_REQUEST = {
    "role": "assistant",
    "content": "",
    "tool_calls": [{
        "id": "mock-call-1",
        "type": "function",
        "function": {"name": "calculator", "arguments": '{"expression": "12 * 7"}'},
    }],
}

messages = [{"role": "user", "content": "What is 12 * 7? Use the calculator."}]

def request_tool_call(messages):
    """Return the assistant message as a plain dict, live or mocked."""
    if client is None:
        return MOCK_TOOL_REQUEST
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=messages,
            tools=[calculator_tool],
            temperature=0,
            max_tokens=400,
            extra_body={"reasoning": {"effort": "low", "exclude": False},
                        "provider": {"require_parameters": True}},
        )
        return response.choices[0].message.model_dump(exclude_none=True)
    except Exception as exc:
        print("Live call failed, using the mock request ->", type(exc).__name__, exc)
        return MOCK_TOOL_REQUEST

assistant_message = request_tool_call(messages)
tool_calls = assistant_message.get("tool_calls") or []

print("Assistant text content :", repr(assistant_message.get("content")))
print("Number of tool requests:", len(tool_calls))
for call in tool_calls:
    print("  id       :", call["id"])
    print("  tool name:", call["function"]["name"])
    print("  arguments:", call["function"]["arguments"], "  <- note: a STRING, not a dict")
print()
print("Has anything been calculated yet? No. This is a request, not a result.")

### Step 5 — Validate the arguments, then execute

Two checks before any Python runs: is this a tool we actually offer, and are the arguments
acceptable? Only then do we call the function.

In [ ]:
AVAILABLE_TOOLS = {"calculator": project_calculator}

def run_tool_call(call):
    """Validate and execute one tool request. Returns the observation text."""
    name = call["function"]["name"]
    if name not in AVAILABLE_TOOLS:
        return f"Tool error: unknown tool '{name}'"          # the model invented a name
    try:
        raw_arguments = json.loads(call["function"]["arguments"])   # string -> dict
        arguments = CalculatorArguments.model_validate(raw_arguments)
    except (json.JSONDecodeError, ValidationError) as error:
        return f"Tool error: bad arguments ({error})"
    try:
        return AVAILABLE_TOOLS[name](arguments.expression)
    except Exception as error:
        return f"Tool error: {error}"

call = tool_calls[0]
observation = run_tool_call(call)
print("tool  :", call["function"]["name"])
print("input :", call["function"]["arguments"])
print("output:", observation)

### Step 6 — Return the observation to the model

The conversation must now contain three things in order: our question, the assistant's
tool request, and a `tool` message carrying the result. The `tool_call_id` is what links
the answer to the question — leave it out and the provider rejects the request.

In [ ]:
messages.append(assistant_message)                       # what the model asked for
messages.append({                                        # what our code observed
    "role": "tool",
    "tool_call_id": call["id"],
    "content": observation,
})

print("Conversation now has", len(messages), "messages:")
for index, message in enumerate(messages):
    requested = [c["function"]["name"] for c in (message.get("tool_calls") or [])]
    print(f"  {index}. role={message['role']:9} tool_requests={requested} content={str(message.get('content'))[:60]!r}")

def final_answer(messages):
    if client is None:
        return f"The calculator returned {observation}, so 12 * 7 = {observation}."
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL, messages=messages, tools=[calculator_tool],
            temperature=0, max_tokens=300,
            extra_body={"reasoning": {"effort": "low", "exclude": True}},
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("Live call failed, using the mock answer ->", type(exc).__name__, exc)
        return f"The calculator returned {observation}."

print()
print("Final answer:", final_answer(messages))

### Step 7 — Break it: an argument our schema refuses

A model can send an empty expression, a 500-character one, or Python code. The tool schema
describes the *shape*; our Pydantic model and the calculator itself decide what is
actually allowed. We prove here that the function is never reached.

In [ ]:
executions = {"count": 0}

def counted_calculator(expression):
    executions["count"] += 1                 # increments ONLY if we really execute
    return project_calculator(expression)

AVAILABLE_TOOLS["calculator"] = counted_calculator

bad_requests = [
    {"id": "b1", "function": {"name": "calculator", "arguments": '{"expression": ""}'}},
    {"id": "b2", "function": {"name": "calculator", "arguments": '{"expression": "' + "9" * 150 + '"}'}},
    {"id": "b3", "function": {"name": "calculator", "arguments": '{"wrong_key": "12 * 7"}'}},
    {"id": "b4", "function": {"name": "calculator", "arguments": 'not json at all'}},
    {"id": "b5", "function": {"name": "calculator", "arguments": '{"expression": "__import__(\'os\').getcwd()"}'}},
    {"id": "b6", "function": {"name": "delete_all_files", "arguments": "{}"}},
]

for request in bad_requests:
    # Validation errors are multi-line; squash them so the table stays readable.
    message = " ".join(run_tool_call(request).split())
    print(f"{request['id']}: {message[:105]}")

print()
print("Times the calculator function actually ran:", executions["count"])
print("Only b5 reached it (its shape was valid) and the AST evaluator refused the code.")
AVAILABLE_TOOLS["calculator"] = project_calculator     # restore

### Checkpoint

**1. The model returns `tool_calls` naming `delete_all_files`. What happens?**

<details><summary>Show answer</summary>

Nothing — unless your code chooses to make it happen. A tool call is generated text. In
Step 5 the lookup `if name not in AVAILABLE_TOOLS` returns a `Tool error: unknown tool`
observation, and the model reads that on its next turn. This is why the registry, not the
model, is the security boundary.

</details>

**2. Why validate arguments with Pydantic when the tool schema already said `expression` is a string?**

<details><summary>Show answer</summary>

The schema is a description sent to the provider; it is advice, and outside strict mode a
model can and does deviate. Even inside strict mode "a string" says nothing about length or
content. The Pydantic model runs on **your** machine, on the data that actually arrived,
and it is the last thing between a model's suggestion and your function. Test `b3` and `b4`
above show requests that never even parse.

</details>

### Recap

- **Limitation we saw:** a model cannot execute anything, so on its own it can only guess
  at arithmetic and outside facts.
- **Layer we added:** a tool schema, a request/validate/execute boundary, and a `tool`
  message that carries the observation back into the conversation.
- **Evidence it worked:** `12 * 7` was answered from a real calculation, and six malformed
  requests produced error observations while the counter proved the function ran once.